# Stage 9a v3 — d=2 + lambda_ctc=0.5 (spot check + conditional 5-fold)

Tests whether **combining the two best-performing ablation knobs compounds**:
- d=2 alone (vs d=3): fold-0 CER 0.5574 (Δ = -0.0100, ≥2σ winner)
- lambda=0.5 alone (vs lambda=0.3): fold-0 CER 0.5627 (Δ = -0.0047, within seed noise)
- **d=2 + lambda=0.5 combined**: unknown — this is the question.

Two-pass workflow in one notebook:
1. **First pass** (`RUN_FULL_5FOLD = False`): Cell 5 trains fold 0 only.  Cell 6 prints the verdict.  ~50 min on T4.
2. **Second pass** (`RUN_FULL_5FOLD = True`, only if verdict was favourable): re-run the kernel.  Fold 0 is skipped via resume, Cells 7+ train folds 1-4.  ~3 h on T4.

## Decision rule for the spot check (Cell 6)

| fold-0 CER at d=2+lambda=0.5 | What it means | Recommended next step |
|---|---|---|
| ≤ **0.5486** (beats d=2-alone by ≥2σ) | ✅ Compounding effect present | Set `RUN_FULL_5FOLD = True`, commit, run 5-fold (~3 h) |
| 0.5486 < CER ≤ 0.5574 | ⚖ Marginal — combines like d=2 alone | Skip 5-fold; use existing `stage9a_v2` (d=2+lambda=0.3) result |
| > 0.5574 | ❌ Regressed past d=2 alone | lambda=0.5 actively hurts at d=2; stay on lambda=0.3 |

## Variant naming

Variant = `stage9a_v3` for ALL folds — fold-0 spot check is just fold 0 of the eventual 5-fold sweep, so the second-pass resume logic naturally picks up where the first pass left off.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate cache + manifest + prior results

In [ ]:
import os, glob, json, shutil
def _first(pattern):
    m = (glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True))
    return m[0] if m else None

CACHE_PATH        = _first('skeleton_features_t32.pt')
CV_MANIFEST_FOUND = _first('subject_cv5.json')
RESULTS_FOUND     = _first('stage9a_v3_results.json')

OUT_CACHE    = CACHE_PATH        or '/kaggle/working/skeleton_features_t32.pt'
OUT_MANIFEST = CV_MANIFEST_FOUND or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage9a_v3_results.json'
if RESULTS_FOUND and not os.path.exists(RESULTS_PATH):
    shutil.copy(RESULTS_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)

for lbl, p in [('cache', OUT_CACHE), ('manifest', OUT_MANIFEST), ('results', RESULTS_PATH)]:
    print(f'{lbl:<10s}: {p}  (exists={os.path.exists(p)})')

## Cell 3 — Config + flag

In [ ]:
import logging, random
import numpy as np
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage9a_v3.log'))])

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# ============ ONLY KNOB YOU MAY FLIP ON SECOND PASS ============
RUN_FULL_5FOLD = False        # set True after Cell 6 spot-check verdict is good
# ===============================================================

# Locked except d=2 and lambda=0.5.
T_NATIVE     = 32
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 32
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
NUM_EPOCHS   = 80
WARMUP_PCT   = 0.05
SEED         = 42
DEC_N_HEADS  = 4

# The two ablation winners stacked:
DEC_N_LAYERS = 2          # ablation winner (vs d=3)
LAMBDA_CTC   = 0.5        # within-noise winner of the lambda sweep
VARIANT_NAME = 'stage9a_v3'

FOLDS_FULL = list(range(5))
FOLDS_SPOT = [0]
FOLDS      = FOLDS_FULL if RUN_FULL_5FOLD else FOLDS_SPOT

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
                      weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                      num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device          : {cfg.device}')
print(f'Variant         : {VARIANT_NAME}')
print(f'Decoder layers  : {DEC_N_LAYERS}  (Stage 9a used 3)')
print(f'lambda_ctc      : {LAMBDA_CTC}    (Stage 9a used 0.3)')
print(f'RUN_FULL_5FOLD  : {RUN_FULL_5FOLD}')
print(f'Folds to train  : {FOLDS}')

## Cell 4 — Load cache + manifest (build only if missing)

In [ ]:
if not os.path.exists(OUT_CACHE):
    from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
    from wita_v2.datasets.skeleton_cache  import extract_skeleton_features
    print('Building skeleton cache from scratch...')
    samples = stream_and_index_with_subjects(cfg)
    extract_skeleton_features(samples=samples, out_path=OUT_CACHE,
                              T_native=T_NATIVE, dtype=torch.float16)
cache = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
print(f'cache: {len(cache["feats"])} clips, '
      f'detect_rate={cache.get("frame_detect_rate",0)*100:.1f}%')

if not os.path.exists(OUT_MANIFEST):
    from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest
    fake = [(b'', cache['labels'][i], cache['subjects'][i])
            for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, OUT_MANIFEST)
else:
    from wita_v2.datasets.cv_splits import load_cv5_manifest
    manifest = load_cv5_manifest(OUT_MANIFEST)
print(f'manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')

## Cell 5 — Train selected folds (fold 0 only on first pass, all 5 on second)

Resume-aware: fold 0 trained in pass 1 is skipped automatically in pass 2.

In [ ]:
from wita_v2.training.stage9_train   import train_one_fold
from wita_v2.datasets.cv_splits       import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} folds already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()

for fold in FOLDS:
    if (fold, VARIANT_NAME) in completed:
        print(f'[skip] fold {fold} already done'); continue
    train_idx, val_idx = fold_indices(manifest, fold, cache['subjects'])
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=fold, variant=VARIANT_NAME,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
        dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
        lambda_ctc=LAMBDA_CTC, transform=train_aug, seed=SEED,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  fold {fold}: best CER = {result["best_val_cer"]:.4f}\n')
print(f'\n{len(all_results)}/{len(FOLDS_FULL)} folds done so far.')

## Cell 6 — Spot-check verdict (always runs; consults fold 0 only)

In [ ]:
with open(RESULTS_PATH) as f:
    all_results = json.load(f)
by_fold = {r['fold']: r for r in all_results if r['variant'] == VARIANT_NAME}

# Reference numbers from the Stage 9a ablation matrix (fold 0).
ABL_CONTROL_D3_L03 = 0.5674     # d=3, lambda=0.3 (l03_s0)
ABL_D2_L03         = 0.5574     # d=2, lambda=0.3 (d2 ablation winner)
ABL_D3_L05         = 0.5627     # d=3, lambda=0.5 (within noise)
TWO_SIGMA          = 0.0088     # from the l03 seed cluster

if 0 not in by_fold:
    print('Fold 0 spot check not yet done — run Cell 5 first.')
else:
    v3_fold0 = by_fold[0]['best_val_cer']
    print('=== Fold-0 spot check: d=2 + lambda_ctc=0.5 ===\n')
    print(f'  d=2 + lambda=0.5 (v3 here)    : {v3_fold0:.4f}')
    print(f'  d=2 + lambda=0.3 (ablation)   : {ABL_D2_L03:.4f}')
    print(f'  d=3 + lambda=0.5 (ablation)   : {ABL_D3_L05:.4f}')
    print(f'  d=3 + lambda=0.3 (control)    : {ABL_CONTROL_D3_L03:.4f}')
    print(f'  2σ threshold                  : ±{TWO_SIGMA:.4f}\n')

    delta_vs_d2  = v3_fold0 - ABL_D2_L03
    delta_vs_l05 = v3_fold0 - ABL_D3_L05
    delta_vs_ctl = v3_fold0 - ABL_CONTROL_D3_L03
    print(f'  Δ vs d=2-alone   : {delta_vs_d2:+.4f}')
    print(f'  Δ vs lambda=0.5-alone: {delta_vs_l05:+.4f}')
    print(f'  Δ vs control     : {delta_vs_ctl:+.4f}\n')

    if delta_vs_d2 <= -TWO_SIGMA:
        print(f'  ✅ COMPOUNDING (beats d=2 alone by ≥2σ).')
        print('     Set RUN_FULL_5FOLD = True in Cell 3 and re-run the kernel.')
        print(f'     Expected 5-fold full mean: ~0.470 ± 0.07.')
    elif delta_vs_d2 <= 0:
        print(f'  ⚖  MARGINAL (within ±2σ of d=2 alone).')
        print('     lambda=0.5 contribution is washed out by d=2.  Recommendation: stay on')
        print('     stage9a_v2 (d=2 + lambda=0.3) for the 5-fold validation.')
    else:
        print(f'  ❌ REGRESSED past d=2 alone.')
        print('     lambda=0.5 actively hurts at d=2 — revert to lambda=0.3.')

## Cell 7 — Aggregate + verdict (only fires after all 5 folds are done)

Skipped silently when fewer than 5 folds are complete (pass-1 case).

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
from wita_v2.reports.template.stripped_cohort import dual_cohort_summary

if len(by_fold) < 5:
    print(f'Only {len(by_fold)}/5 folds done — re-run with RUN_FULL_5FOLD=True after a favourable spot check.')
else:
    STAGE9A_FOLD = {0: 0.5681, 1: 0.5345, 2: 0.5054, 3: 0.4374, 4: 0.3991}
    STAGE9A_FULL_MEAN     = 0.4889
    STAGE9A_STRIPPED_MEAN = 0.4775

    print(' fold    Stage 9a    Stage 9a v3 (d=2 + λ=0.5)    Δ (v3 - 9a)    best ep')
    for f in FOLDS_FULL:
        r = by_fold[f]
        d = r['best_val_cer'] - STAGE9A_FOLD[f]
        tag = '✅' if d <= -TWO_SIGMA else ('❌' if d >= TWO_SIGMA else '⚖')
        print(f'  {f:>2d}    {STAGE9A_FOLD[f]:.4f}      {r["best_val_cer"]:.4f}                  '
              f'{d:+.4f} {tag}    ep {r["best_epoch"]}')

    s = dual_cohort_summary(RESULTS_PATH, VARIANT_NAME)
    print(f'\n  Stage 9a v3 full     : {s["full_mean"]:.4f} ± {s["full_std"]:.4f}')
    print(f'  Stage 9a v3 stripped : {s["stripped_mean"]:.4f} ± {s["stripped_std"]:.4f}')
    print(f'  Stage 9a  full       : {STAGE9A_FULL_MEAN:.4f}   stripped : {STAGE9A_STRIPPED_MEAN:.4f}')
    full_delta = s['full_mean'] - STAGE9A_FULL_MEAN
    print(f'  Δ full mean (v3 - 9a): {full_delta:+.4f}')

    a = np.array([STAGE9A_FOLD[f]            for f in FOLDS_FULL])
    b = np.array([by_fold[f]['best_val_cer'] for f in FOLDS_FULL])
    try:
        W, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
        print(f'  Paired Wilcoxon: W={W:.2f}  p≈{p:.4f}  n={len(a)}')
    except ValueError as e:
        print(f'  Wilcoxon n/a: {e}')

    print('\n=== Stage 9a v3 verdict ===')
    if s['full_mean'] <= 0.460:
        print(f'  ✅ STRETCH ({s["full_mean"]:.4f} ≤ 0.460) — d=2+λ=0.5 is a real ~3 pt win.')
        print('     Adopt v3 as the new Stage 9 headline.')
    elif s['full_mean'] <= 0.475:
        print(f'  ✅ HEADLINE ({s["full_mean"]:.4f} ≤ 0.475) — d=2+λ=0.5 is a real ~1.5 pt win.')
        print('     Worth adopting; chain Stage 9b LM rescoring off this kernel.')
    elif abs(full_delta) <= TWO_SIGMA:
        print(f'  ⚖  TIE ({full_delta:+.4f} within ±2σ).')
        print('     Stage 9a (d=3, λ=0.3) and v3 are equivalent at 5-fold scale.')
    else:
        print(f'  ❌ REGRESS ({full_delta:+.4f}).')
        print('     d=2+λ=0.5 underfits at full scale; revert to Stage 9a headline.')

## Cell 8 — Per-signer scatter (v3 vs Stage 9a)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if len(by_fold) < 5:
    print('Skipping per-signer scatter until all 5 folds are done.')
else:
    S9A_RESULTS = _first('stage9a_results.json')
    if not S9A_RESULTS:
        print('Stage 9a per-signer JSON not attached — skipping scatter.')
    else:
        with open(S9A_RESULTS) as f:
            s9a_all = json.load(f)
        s9a_ps = {}
        for r in s9a_all:
            if r.get('variant') == 'stage9a':
                s9a_ps.update(r.get('best_per_signer_val_cer', {}) or {})
        v3_ps = {}
        for r in all_results:
            if r['variant'] == VARIANT_NAME:
                v3_ps.update(r.get('best_per_signer_val_cer', {}) or {})
        signers = sorted(set(s9a_ps) | set(v3_ps))
        DATASET_LIMIT = ['PHW', 'KIM']
        MODEL_HARD    = ['PJH','SYB','KJM','KNY','LKS','YMG']
        def _c(s):
            if s in DATASET_LIMIT: return '#7f7f7f'
            if s in MODEL_HARD:    return '#d62728'
            return '#1f77b4'
        xs = [s9a_ps.get(s, float('nan')) for s in signers]
        ys = [v3_ps.get(s, float('nan')) for s in signers]
        colors = [_c(s) for s in signers]
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        lo, hi = 0.2, 1.0
        ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='y = x')
        ax.scatter(xs, ys, c=colors, s=48, edgecolor='black', linewidths=0.4)
        for s, x, y in zip(signers, xs, ys):
            ax.annotate(s, (x, y), fontsize=6, alpha=0.7,
                        xytext=(3, 3), textcoords='offset points')
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
        ax.set_xlabel('Stage 9a (d=3, λ=0.3)')
        ax.set_ylabel('Stage 9a v3 (d=2, λ=0.5)')
        ax.set_title('Per-signer val CER  (below y=x → v3 wins)')
        ax.legend(frameon=False, loc='upper left')
        ax.grid(True, linestyle=':', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, 'stage9a_v3_per_signer.png'), dpi=140)
        plt.show()

## Cell 9 — Commit reminder

**Save Version → Save & Run All** to preserve:
- `stage9a_v3_results.json`
- `checkpoints/stage9a_fold*_stage9a_v3_best.pt` (5 if you ran the 5-fold pass)
- `logs/stage9a_v3.log`
- `logs/stage9a_v3_per_signer.png` (5-fold pass only)

If verdict is HEADLINE or STRETCH at 5-fold, the next Stage 9b LM rescoring kernel should attach this kernel's v3 checkpoints (not the original Stage 9a's).